## Goal

The main goal of this notebook is to replicate the results from the plot below. Which is taken from the paper -> Analysis of CNN for rPPG estimation

In [6]:
import yaml
import os
import sys
sys.path.append("/homes/bacharya/rPPG-Toolbox")
from config import get_config
from dataset import data_loader
import argparse
from torch.utils.data import DataLoader
from notebooks import syncpos


In [7]:
config_file_path = "/homes/bacharya/rPPG-Toolbox/configs/train_configs/PURE_PURE_PURE_DEEPPHYS_FINGER_PPG.yaml"
args = argparse.Namespace()
args.config_file = config_file_path
args.FOLD = 0
config = get_config(args)


=> Merging a config file from /homes/bacharya/rPPG-Toolbox/configs/train_configs/PURE_PURE_PURE_DEEPPHYS_FINGER_PPG.yaml


In [8]:
#Load the pure dataset
train_loader = data_loader.PURELoader.PURELoader( 
                name="train",
                data_path=config.TRAIN.DATA.DATA_PATH,
                config_data=config.TRAIN.DATA,
                model = config.MODEL.NAME
            )

test_loader = data_loader.PURELoader.PURELoader(
                name="test",
                data_path=config.TEST.DATA.DATA_PATH,
                config_data=config.TEST.DATA,
                model=config.MODEL.NAME
            )

# Create your data loaders
train_dataloader = DataLoader(
        dataset=train_loader,
        num_workers=2,
        batch_size=config.TRAIN.BATCH_SIZE,
        shuffle=False
)

test_dataloader = DataLoader(
        dataset=test_loader,
        num_workers=2,
        batch_size=config.INFERENCE.BATCH_SIZE,
        shuffle=False
)

# Create your model
model = syncpos.DeepPhysOnlyMotionTrainer.DeepPhysOnlyMotionTrainer(config, {"train": train_dataloader})

# # Initialize the custom trainer
# trainer = ConvergenceThenFixedEpochsTrainer(
#     model=model,
#     train_dataloader=train_dataloader,
#     val_dataloader=val_dataloader,
#     test_dataloader=test_dataloader,
#     config=config,
#     patience=10,
#     post_convergence_epochs=15
# )

# # Train the model
# converged_model_path, post_convergence_model_paths = trainer.train()

# # Define your custom evaluation metric
# def custom_metric(model, test_dataloader):
#     # Implement your custom evaluation logic here
#     # For example, calculate precision, recall, F1, or any domain-specific metric
#     # Return a score where higher is better
#     score = ...
#     return score

# # Evaluate models with custom metric
# best_model_path, best_score, all_scores = trainer.evaluate_post_convergence_models(custom_metric)

# # Final evaluation on best model
# final_results = trainer.final_evaluation(best_model_path)


Cached Data Path /data/rppg_14_pure_video_nt_lab/processed/sync-exp/PURE_SizeW64_SizeH64_ClipLength1_DataTypeDiffNormalized_DataAugNone_LabelTypeDiffNormalized_Crop_faceTrue_BackendHC_Large_boxTrue_Large_size1.5_Dyamic_DetTrue_det_len1_Median_face_boxFalse_PSEUDO_LABELFalse

File List Path /data/rppg_14_pure_video_nt_lab/processed/sync-exp/DataFileLists/PURE_SizeW64_SizeH64_ClipLength1_DataTypeDiffNormalized_DataAugNone_LabelTypeDiffNormalized_Crop_faceTrue_BackendHC_Large_boxTrue_Large_size1.5_Dyamic_DetTrue_det_len1_Median_face_boxFalse_PSEUDO_LABELFalse_0.0_0.5.csv
 train Preprocessed Dataset Length: 62645

Cached Data Path /data/rppg_14_pure_video_nt_lab/processed/sync-exp/PURE_SizeW64_SizeH64_ClipLength1_DataTypeDiffNormalized_DataAugNone_LabelTypeDiffNormalized_Crop_faceFalse_BackendHC_Large_boxTrue_Large_size1.5_Dyamic_DetTrue_det_len1_Median_face_boxFalse_PSEUDO_LABELFalse

File List Path /data/rppg_14_pure_video_nt_lab/processed/sync-exp/DataFileLists/PURE_SizeW64_SizeH64_Clip

Exception: Unsupported image size

In [9]:
import os
import torch
import torch.optim as optim
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger
from collections import deque

class ConvergenceThenFixedEpochsTrainer:
    def __init__(self, model, train_dataloader, val_dataloader, test_dataloader, 
                 config, patience, post_convergence_epochs=15):
        """
        Initialize trainer that first trains until convergence, then for a fixed number of epochs.
        
        Args:
            model: Your PyTorch Lightning model
            train_dataloader: DataLoader for training data
            val_dataloader: DataLoader for validation data
            test_dataloader: DataLoader for test data
            config: Configuration object with training parameters
            patience: Number of epochs to wait for improvement before declaring convergence
            post_convergence_epochs: Number of epochs to train after convergence
        """
        self.model = model
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.test_dataloader = test_dataloader
        self.config = config
        self.patience = patience
        self.post_convergence_epochs = post_convergence_epochs
        self.save_dir = os.path.join(config.MODEL.MODEL_DIR, "post_convergence_models")
        self.model_file_name = config.TRAIN.MODEL_FILE_NAME
        
        # Create save directory if it doesn't exist
        if not os.path.exists(self.save_dir):
            os.makedirs(self.save_dir)
            
    def train(self):
        """Main training method that handles both convergence and post-convergence phases."""
        # Phase 1: Train until convergence
        print("Phase 1: Training until convergence...")
        
        # Setup early stopping callback
        early_stop_callback = EarlyStopping(
            monitor="val_loss",
            min_delta=0.001,
            patience=self.patience,
            verbose=True,
            mode="min"
        )
        
        # Setup logger for phase 1
        logger_phase1 = TensorBoardLogger(
            save_dir=os.path.join(self.config.LOG.PATH, "phase1"),
            name=self.model_file_name
        )
        
        # Create trainer for phase 1
        trainer_phase1 = pl.Trainer(
            max_epochs=50,  # Set a high number, will stop early due to early stopping
            callbacks=[early_stop_callback],
            logger=logger_phase1,
            accelerator='gpu' if torch.cuda.is_available() else 'cpu',
            devices=1
        )
        
        # Train until convergence
        trainer_phase1.fit(self.model, self.train_dataloader, self.val_dataloader)
        
        # Save converged model
        converged_model_path = os.path.join(self.save_dir, f"{self.model_file_name}_converged.pth")
        torch.save(self.model.state_dict(), converged_model_path)
        print(f"Converged model saved to {converged_model_path}")
        
        # Phase 2: Continue training for fixed number of epochs
        print(f"Phase 2: Training for additional {self.post_convergence_epochs} epochs...")
        
        # Setup checkpoint callback to save each epoch
        checkpoint_callback = ModelCheckpoint(
            dirpath=self.save_dir,
            filename=f"{self.model_file_name}_post_convergence_epoch_{{epoch}}",
            save_top_k=-1,  # Save all models
            every_n_epochs=1
        )
        
        # Setup logger for phase 2
        logger_phase2 = TensorBoardLogger(
            save_dir=os.path.join(self.config.LOG.PATH, "phase2"),
            name=self.model_file_name
        )
        
        # Create trainer for phase 2
        trainer_phase2 = pl.Trainer(
            max_epochs=self.post_convergence_epochs,
            callbacks=[checkpoint_callback],
            logger=logger_phase2,
            accelerator='gpu' if torch.cuda.is_available() else 'cpu',
            devices=1
        )
        
        # Continue training for fixed number of epochs
        trainer_phase2.fit(self.model, self.train_dataloader, self.val_dataloader)
        
        # Return paths to all saved models for later evaluation
        model_paths = [
            os.path.join(self.save_dir, f"{self.model_file_name}_post_convergence_epoch_{i}.ckpt") 
            for i in range(1, self.post_convergence_epochs + 1)
        ]
        
        return converged_model_path, model_paths
    
    def evaluate_post_convergence_models(self, custom_metric_function):
        """
        Evaluate all post-convergence models using a custom metric function.
        
        Args:
            custom_metric_function: Function that takes model and test_dataloader and returns a metric score
            
        Returns:
            best_model_path: Path to the model with the best score
            best_score: The best score achieved
            all_scores: Dictionary mapping model paths to their scores
        """
        print("Evaluating post-convergence models with custom metric...")
        
        # Get list of all post-convergence model checkpoints
        model_paths = [
            os.path.join(self.save_dir, f) 
            for f in os.listdir(self.save_dir) 
            if f.startswith(f"{self.model_file_name}_post_convergence_epoch_") and f.endswith(".ckpt")
        ]
        
        # Sort by epoch number
        model_paths.sort(key=lambda x: int(x.split("_")[-1].split(".")[0]))
        
        # Evaluate each model
        scores = {}
        best_score = float('-inf')
        best_model_path = None
        
        for path in model_paths:
            # Load model weights
            self.model.load_from_checkpoint(path)
            
            # Evaluate with custom metric
            score = custom_metric_function(self.model, self.test_dataloader)
            scores[path] = score
            
            print(f"Model {path}: score = {score}")
            
            # Track best model
            if score > best_score:
                best_score = score
                best_model_path = path
        
        print(f"Best model: {best_model_path} with score {best_score}")
        
        return best_model_path, best_score, scores
    
    def final_evaluation(self, best_model_path):
        """
        Perform final evaluation on the best model.
        
        Args:
            best_model_path: Path to the best model checkpoint
        
        Returns:
            test_results: Results from the test evaluation
        """
        print(f"Performing final evaluation on {best_model_path}...")
        
        # Load best model
        self.model = self.model.load_from_checkpoint(best_model_path)
        
        # Setup trainer for testing
        trainer = pl.Trainer(
            accelerator='gpu' if torch.cuda.is_available() else 'cpu',
            devices=1
        )
        
        # Test the model
        test_results = trainer.test(self.model, self.test_dataloader)
        
        return test_results